# Daily ATP Dataset Sync & Schedule Prediction Loader

This notebook pulls the **latest unversioned** daily release of `dissfya/atp-tennis-2000-2023daily-pull` from Kaggle and triggers the automated `atp_daily_sync` pipeline to refresh rolling player states and predict newly published ATP matches.

In [ ]:
# Install dependencies as needed:
# pip install -r requirements.txt
import os
import pandas as pd
import kagglehub

# 1. Fetch latest unversioned daily pull (never pin /versions/XXXX for daily inference)
dataset_dir = kagglehub.dataset_download(
    "dissfya/atp-tennis-2000-2023daily-pull",
    path="atp_tennis.csv",
    force_download=True,
)
csv_path = dataset_dir if os.path.isfile(dataset_dir) else os.path.join(dataset_dir, "atp_tennis.csv")
df = pd.read_csv(csv_path)
print(f"Loaded {len(df):,} rows through {df['Date'].max()}")
print(df.tail(5))

In [ ]:
# 2. Run incremental feature refresh + published ATP schedule prediction
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from pipeline.atp_daily_sync import run_daily_pipeline

result = run_daily_pipeline()
print("Sync status:", result["sync"])
print("Scheduled fixtures predicted:", result["predictions"]["fixture_count"])
pd.DataFrame(result["predictions"]["fixtures"]).head(10)